<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/v2s_fullEdited.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [3]:
import os
import shutil
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S, ConvNeXtTiny
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input


In [4]:
BASE = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"

MODEL_SAVE_PATH = "/drive/MyDrive/dermoscopy_sota_model.keras"


In [5]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

shutil.copytree(IMG_SRC, BASE)

'/content/newdata'

In [6]:
batch_size = 16
image_size = 256

In [7]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
])

In [8]:
def load(path, shuffle):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=shuffle
    )
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = load(f"{BASE}/train", True)
val_ds = load(f"{BASE}/valid", False)
test_ds = load(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [9]:
loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=2.0,
    label_smoothing=0.05
)

In [10]:
def create_sota_model():

    inputs = layers.Input(shape=(image_size, image_size, 3))

    # ---------------- RGB AUGMENT ----------------
    x = augment(inputs)
    x = preprocess_input(x)

    # =====================================================
    # BACKBONE 1 - EfficientNetV2S
    # =====================================================
    eff = EfficientNetV2S(include_top=False, weights="imagenet", input_tensor=x)

    for layer in eff.layers[:-200]:
        layer.trainable = False

    eff_feat = eff.output

    # =====================================================
    # BACKBONE 2 - ConvNeXt Tiny
    # =====================================================
    cx = ConvNeXtTiny(include_top=False, weights="imagenet", input_tensor=x)

    for layer in cx.layers[:-120]:
        layer.trainable = False

    cx_feat = cx.output

    # =====================================================
    # FEATURE FUSION
    # =====================================================
    fused = layers.Concatenate()([eff_feat, cx_feat])

    fused = layers.Conv2D(512, 1, activation="relu")(fused)

    # =====================================================
    # ATTENTION BLOCK (SE STYLE)
    # =====================================================
    se = layers.GlobalAveragePooling2D()(fused)
    se = layers.Dense(256, activation="relu")(se)
    se = layers.Dense(512, activation="sigmoid")(se)

    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Multiply()([fused, se])

    # =====================================================
    # CLASSIFIER HEAD
    # =====================================================
    x = layers.Dense(256, activation="relu")(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(2, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
        loss=loss_fn,
        metrics=["accuracy"]
    )

    return model

In [11]:
model = create_sota_model()
model.summary()

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 256, 256,  │          0 │ input_layer[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 256, 256,  │          0 │ sequential[0][0]  │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 128, 128,  │        648 │ rescaling[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 128, 128,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 128, 128,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 128, 128,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 128, 128,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 128, 128,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 128, 128,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 128, 128,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 128, 128,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 128, 128,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 128, 128,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 128, 128,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 64, 64,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 64, 64,    │        384 │ block2a_expand_c

 Total params: 49,628,994 (189.32 MB)

 Trainable params: 42,567,490 (162.38 MB)

 Non-trainable params: 7,061,504 (26.94 MB)

In [12]:
checkpoint = ModelCheckpoint(
    CHECKPOINT_DIR + "/best_sota.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-7,
    verbose=1
)

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[checkpoint, early, lr]
)

Epoch 1/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 954ms/step - accuracy: 0.7517 - loss: 0.0578
Epoch 1: val_accuracy improved from None to 0.81618, saving model to /drive/MyDrive/checkpoints/best_sota.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_sota.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 613s 1s/step - accuracy: 0.7802 - loss: 0.0527 - val_accuracy: 0.8162 - val_loss: 0.0298 - learning_rate: 3.0000e-05
Epoch 2/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 944ms/step - accuracy: 0.8209 - loss: 0.0407
Epoch 2: val_accuracy improved from 0.81618 to 0.85315, saving model to /drive/MyDrive/checkpoints/best_sota.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_sota.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 498s 994ms/step - accuracy: 0.8291 - loss: 0.0391 - val_accuracy: 0.8531 - val_loss: 0.0295 - learning_rate: 3.0000e-05
Epoch 3/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 954ms/step - accuracy: 0.8347 - loss: 0.0350
Epoch 3: val_accuracy improved from 0.85315 to 0.88312, sa

In [14]:
loss, acc = model.evaluate(test_ds)

print("\n====================")
print("FINAL RESULTS")
print("====================")
print("Test Accuracy:", acc)

63/63 ━━━━━━━━━━━━━━━━━━━━ 22s 339ms/step - accuracy: 0.8882 - loss: 0.0242

FINAL RESULTS
Test Accuracy: 0.8882235288619995
